# Phân tích ANOVA cho NSL-KDD

Notebook này trả lời bốn câu hỏi nghiên cứu về `count`, `serror_rate`, `src_bytes` và `duration` giữa năm nhóm Normal, DoS, Probe, R2L, U2R. Đây là phân tích thống kê suy diễn trên **tập train sạch ở đơn vị gốc**, không phải huấn luyện Machine Learning; test set được giữ riêng.

## Dữ liệu, giả thuyết và ý nghĩa của ANOVA

Với từng đặc trưng, đặt $H_0: \mu_{Normal}=\mu_{DoS}=\mu_{Probe}=\mu_{R2L}=\mu_{U2R}$ và $H_1$: có ít nhất một cặp trung bình tổng thể khác nhau, với $\alpha=0.05$.

Thống kê $F$ đo tỷ lệ biến thiên giữa nhóm so với biến thiên trong nhóm; Welch dùng trọng số theo phương sai và bậc tự do xấp xỉ khi phương sai/cỡ mẫu khác nhau. Bậc tự do phản ánh lượng thông tin độc lập dùng để xác định phân phối tham chiếu. p-value là xác suất, dưới $H_0$ và các giả định mô hình, quan sát thống kê ít nhất cực đoan như kết quả hiện tại. Bác bỏ $H_0$ khi p-value đã hiệu chỉnh nhỏ hơn alpha. Không bác bỏ không chứng minh $H_0$ đúng; omnibus có ý nghĩa cũng không có nghĩa mọi cặp nhóm đều khác.

## Kế hoạch phương pháp

RQ1 và RQ2 dùng Welch trên thang gốc để trả lời câu hỏi về mean của `count` và `serror_rate`. RQ2 được xem là xấp xỉ vì dữ liệu dồn ở 0/1. RQ3 và RQ4 dùng Welch trên `log1p` do lệch/cực trị hoặc zero inflation; vì vậy estimand là mean sau biến đổi, không phải mean byte/giây gốc. Brown–Forsythe, thống kê shape, Q–Q và ECDF được đọc cùng nhau; Shapiro trên mẫu tối đa 5.000 không quyết định phương pháp một mình. Games–Howell xử lý 10 cặp trong mỗi RQ, còn Holm xử lý họ 4 omnibus p-value. Kruskal–Wallis chỉ là sensitivity analysis về phân phối theo hạng.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'anova_utils.py').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'anova_analysis'
sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))
from anova_utils import run_analysis


In [ ]:
tables = run_analysis()
tables['data_validation']

## Thống kê mô tả và kiểm tra giả định

Bảng mô tả giữ độ chính xác tính toán và chỉ định rõ t-CI 95% cho mean. Với phân phối lệch/cực trị, CI được xem cùng chẩn đoán. Không xóa hay winsorize outlier. Tính độc lập không thể kiểm chứng vì dữ liệu không có định danh phiên/chiến dịch đầy đủ.

In [ ]:
tables['descriptive_statistics']

In [ ]:
tables['assumption_checks']

## Welch ANOVA, effect size, hậu kiểm và multiple testing

Eta-squared và omega-squared được tính từ tổng bình phương cổ điển trên đúng thang phân tích. Khi kiểm định chính là Welch, đây là effect size **mô tả**, không phải đại lượng suy ra trực tiếp từ Welch F. Không áp dụng ngưỡng nhỏ/vừa/lớn máy móc và không dùng effect size để xếp hạng khả năng dự đoán.

In [ ]:
tables['summary_results']

In [ ]:
tables['posthoc_results']

In [ ]:
tables['sensitivity_analysis']

## RQ4: phân tích hai phần

Phần thứ nhất hỏi xác suất `duration=0` có khác theo nhóm hay không. Phần thứ hai chỉ hỏi, trong các kết nối có `duration>0`, mean `log1p(duration)` có khác hay không. Hai câu hỏi này không thay thế nhau và phần dương không đại diện cho toàn bộ kết nối.

In [ ]:
tables['duration_zero_analysis'], tables['duration_positive_analysis']

## Hình, thảo luận và giới hạn

Mỗi feature có boxplot, mean + CI, ECDF, Q–Q và heatmap hậu kiểm; biến lệch có thêm boxplot log1p. RQ4 mô tả tỷ lệ zero bằng Wilson CI (không thực hiện Chi-square trong phần ANOVA) và phân tích bổ sung có điều kiện trên duration > 0. Caption nêu rõ hình cho biết và không cho biết điều gì. Pipeline còn sinh đúng 5 hình tuyển chọn của báo cáo vào `figures/report` và đồng bộ sang `Latex/ANOVA/figures`. Kết luận chi tiết được sinh vào `results/interpretation.md`. NSL-KDD cũ, mất cân bằng và có thể chứa phụ thuộc giữa bản ghi; ANOVA không chứng minh nguyên nhân tấn công hay xác suất tấn công.

In [ ]:
tables['figure_captions']